# Kujutiste loomise rakenduste arendamine (Azure OpenAI)

Suure keelega mudelitel (LLM) on rohkem funktsioone kui ainult teksti genereerimine. Saad ka tekstikirjeldustest pilte luua. Kujutised kui meediumid on kasulikud MedTechi, arhitektuuri, turismi, mängude arenduse, turunduse ja muude valdkondade jaoks. Selles õppetükis töötame tänapäevaste **GPT Image** mudelitega Microsoft Foundryl.

## Õpieesmärgid

Selle õppetüki lõpus oskad:

- Selgitada, mis on kujutiste genereerimine ja kus seda kasutatakse.
- Mõista `gpt-image` mudelite perekonda ja kuidas see erineb varasematest DALL·E mudelitest.
- Luua kujutiste loomise rakendus ja piltide töötlemine.

## Mis on kujutiste genereerimine?

Kujutiste genereerimise mudelid loovad pilte teksti alusel. Kaasaegsed mudelid nagu `gpt-image` õpivad koolituse käigus teksti ja piltide vahelisi seoseid ning tõmbavad iteratiivselt juhuslikust mürast pildi, mis vastab su tekstipäringule.

Kaks tuntud kujutismudelite peret on:

- **`gpt-image` (OpenAI)** — tänapäevane mudel, mida sellel kursusel kasutatakse. Toetab teksti-pildiks genereerimist ja piltide töötlemist (peatäide põhimõttel toimiv täitmine).
- **Midjourney** — populaarne kolmanda osapoole mudel oma teenuse ja Discordipõhise töövooga.

> Vanemad OpenAI kujutismudelid — **DALL·E 2** ja **DALL·E 3** — on juba aegunud. DALL·E 3 pole enam uute juurutuste jaoks saadaval ning sellised funktsioonid nagu `create_variation` olid ainult DALL·E 2-s. Uute rakenduste jaoks kasuta `gpt-image` mudeleid.

Microsoft Foundryl on **`gpt-image-2`** uusim ja võimekaim kujutismudel ning soovitatav vaikimisi valik. Samuti on laialdaselt saadaval `gpt-image-1.5` ja `gpt-image-1-mini`.

> **Oluline:** `gpt-image` mudelid tagastavad loodud pildi **base64** kujul (`b64_json`), mitte URL-ina. Sinu kood dekodeerib base64 stringi baitideks ja salvestab selle — pilti ei saa URL-ist lihtsalt alla laadida.


## Esimese pildigeneratsiooni rakenduse loomine

Mida on vaja pildigeneratsiooni rakenduse loomiseks? Sul on vaja järgmisi teeke:

- **python-dotenv**, soovitatav on kasutada seda teeki, et hoida oma saladusi *.env* faili sees, eemal koodist.
- **openai**, seda teeki kasutad OpenAI API-ga suhtlemiseks.
- **pillow**, Pythonis piltidega töötamiseks.

Kui seda veel tehtud ei ole, järgi juhiseid lehel [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst), et luua Microsoft Foundry ressurss ja mudel. Vali mudeliks **gpt-image-2** (uuem Azure OpenAI pildimudel; DALL·E on aegunud).

1. Loo fail *.env* järgmise sisuga:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Leia see teave Microsoft Foundry portaali "Deployments" sektsioonist oma ressursi jaoks.



1. Koguge ülaltoodud teegid faili nimega *requirements.txt* järgmiselt:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Seejärel looge virtuaalne keskkond ja installige teegid:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windowsi jaoks kasuta järgmisi käske virtual environment'i loomiseks ja aktiveerimiseks:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Lisa järgmine kood faili nimega *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # Konfigureeri Azure OpenAI (Microsoft Foundry) klient.
    # Pildimudelid vajavad uut API versiooni — kontrolli Foundry dokumentatsiooni, millist versiooni sinu mudel vajab.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Loo pilt, kasutades pildigeneratsiooni API-d
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Sisesta siia oma soovitekst
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # nt "gpt-image-2"
        )
        # Määra kaust salvestatud pildi jaoks
        image_dir = os.path.join(os.curdir, 'images')

        # Kui kaust puudub, loo see
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Algata pilditee (märgi, et failitüüp peaks olema png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-pildimudelid tagastavad pildi base64 formaadis (b64_json), mitte URL-ina
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Kuvage pilt vaikimisi pildivaaturis
        image = Image.open(image_path)
        image.show()

    # püüa kinni erandid
    except BadRequestError as err:
        print(err)

    ```

Vaatame seda koodi lähemalt:

- Esiteks impordime vajalikud teegid, sh OpenAI teegi, dotenv teegi, base64 mooduli ja Pillow teegi.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Seejärel laadime keskkonnamuutujad failist *.env*.

    ```python
    # importige dotenv
    dotenv.load_dotenv()
    ```

- Seejärel konfigureerime Azure OpenAI (Microsoft Foundry) kliendi.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Seejärel genereerime pildi:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Sisestage oma käsu tekst siia
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` mudelid tagastavad pildi **base64** stringina `data[0].b64_json`. Me dekodeerime selle baitideks ja kirjutame faili — allalaadimiseks URL-i pole.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lõpuks avame pildi ja kasutame tavapärast pildivaaturit selle kuvamiseks:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Rohkem üksikasju pildi genereerimise kohta

Vaatame `images.generate` parameetreid:

- **prompt**, on teksti kirjeldus, mida kasutatakse pildi genereerimiseks. Siin on see "Jänes hobusel, hoides kommi, uduises luhal, kus kasvavad nartsissid".
- **size**, on genereeritud pildi suurus (näiteks `1024x1024`, `1536x1024`, `1024x1536` või `"auto"`).
- **n**, on genereeritavate piltide arv. Siin genereerime ühe.
- **model**, on sinu pildimudeli juurutamise nimi (näiteks `gpt-image-2`).

> Pildimudelid ei kasuta `temperature` parameetrit — see on teksti genereerimise kontroll. Mitmekesisuse jaoks kutsu API uuesti; mitmekesisuse vähendamiseks tee kirjeldus täpsemaks.

## Täiendavad pildi genereerimise võimalused

Nägid, kuidas mõne Pythonireaga pilti genereerida. `gpt-image` mudelid võivad ka olemasolevat pilti **redigeerida**. Kui anda pilt, valikuline **mask** (mis märgib piirkonna, mida muuta) ja prompt, saad muuta osa pildist — näiteks lisada jänesele mütsi.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# muudatused tagastatakse ka base64 vormingus
edited_image = base64.b64decode(response.data[0].b64_json)
```

Baaspilt sisaldab ainult jänest; lõplik pilt sisaldab jänese mütsi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
